# AG-02: Customer Support Ticket Triage — Multi-Agent System Exploration & Evaluation

This notebook provides a comprehensive walkthrough of the **Customer Support Ticket Triage** agentic workflow, built using **LangGraph**, specialized agents, tool integrations, and human-in-the-loop oversight.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.graph import triage_app
from src.tools.crm_tool import crm_lookup_tool
from src.tools.policy_retriever import policy_retriever_tool
from src.tools.action_tool import action_engine_tool
from src.evaluate import run_evaluation
print('Environment initialized successfully!')

Environment initialized successfully!


## 1. Inspect Knowledge Base Policies and Vector Index
The system indexes 5 corporate policy documents covering refunds, security, logistics, SLAs, and billing.

In [2]:
print(f'Indexed {len(policy_retriever_tool.chunks)} policy chunks across knowledge base files.')
query = 'What is the autonomous refund limit for damaged merchandise?'
results = policy_retriever_tool.retrieve(query, top_k=2)
for r in results:
    print(f"Policy: {r['policy_id']} | Title: {r['title']} | Score: {r['relevance_score']}")
    print(f"Excerpt: {r['excerpt'][:180]}...\n")

Indexed 29 policy chunks across knowledge base files.
Policy: POL-REF-001 | Title: Refund and Cancellation Policy - 3. Autonomous Refund Authority Limits | Score: 0.285
Excerpt: - **Autonomous Threshold:** Automated agent workflows may approve and disburse refunds up to **$50.00 USD** per order without human intervention, provided that:
  - The customer ac...

Policy: POL-BIL-005 | Title: Billing and Subscription Management Policy - 2. Accidental Renewals and 48-Hour Grace Period | Score: 0.1237
Excerpt: - If a customer contacts support within forty-eight (48) hours of an unwanted automatic renewal and has not logged more than 5 platform actions, they are entitled to an immediate f...



## 2. Customer CRM Exploration
Let us load the mock CRM database and examine customer distributions by tier and fraud risk.

In [3]:
crm_path = project_root / 'data' / 'crm_database.json'
with open(crm_path, 'r', encoding='utf-8') as f:
    customers = json.load(f)['customers']

df_crm = pd.DataFrame(list(customers.values()))[['customer_id', 'name', 'tier', 'account_status', 'lifetime_value', 'fraud_risk_score']]
df_crm

,customer_id,name,tier,account_status,lifetime_value,fraud_risk_score
0,CUST-1001,Sarah Jenkins,STANDARD,ACTIVE,145.5,0.04
1,CUST-1002,Marcus Vance,VIP_PLATINUM,ACTIVE,3450.0,0.02
2,CUST-1003,Elena Rostova,GOLD,FLAGGED_SECURITY,890.0,0.72
3,CUST-1004,David Cho,GOLD,ACTIVE,620.0,0.05
4,CUST-1005,Victoria Sterling,VIP_PLATINUM,ACTIVE,4800.0,0.01
5,CUST-1006,Robert Miller,STANDARD,ACTIVE,75.0,0.18
6,CUST-1007,Amina Al-Mansoor,GOLD,ACTIVE,1150.0,0.03
7,CUST-1008,Kevin Zhang,STANDARD,ACTIVE,180.0,0.08
8,CUST-1009,Chloe Dupont,STANDARD,ACTIVE,45.0,0.02
9,CUST-1010,Anonymous / Suspicious Actor,STANDARD,SUSPICIOUS,0.0,0.91


## 3. End-to-End Execution of Sample Scenarios
We execute Scenario 1 (autonomous low-value refund) and Scenario 2 (consequential high-value hardware return).

In [4]:
scenarios_path = project_root / 'data' / 'test_scenarios.json'
with open(scenarios_path, 'r', encoding='utf-8') as f:
    scenarios = json.load(f)

# Run Scenario 1
res1 = triage_app.invoke(scenarios[0], config={'configurable': {'thread_id': 'nb-1'}})
print('=== SCENARIO 1 (Autonomous) ===')
print(f"Category: {res1['category']} | Requires Approval: {res1['requires_human_approval']} | Action: {res1['executed_action']}")
print(f"Final Response Excerpt:\n{res1['final_response'][:250]}...\n")

# Run Scenario 2
res2 = triage_app.invoke(scenarios[1], config={'configurable': {'thread_id': 'nb-2'}})
print('=== SCENARIO 2 (Consequential / Human Gate) ===')
print(f"Category: {res2['category']} | Requires Approval: {res2['requires_human_approval']} | Reason: {res2['approval_reason']}")
print(f"Risk Score: {res2['risk_score']} | Supervisor Decision: {res2['human_decision']}")

=== SCENARIO 1 (Autonomous) ===
Category: Billing & Refunds | Requires Approval: False | Action: issue_refund
Final Response Excerpt:
Hello Sarah,

Thank you for contacting NovaTech Support. We are very sorry to hear that your items in order ORD-8821 arrived damaged. Under our 30-Day Customer Satisfaction Guarantee (POL-REF-001), I have processed a full refund of $28.50 back to you...

=== SCENARIO 2 (Consequential / Human Gate) ===
Category: Billing & Refunds | Requires Approval: True | Reason: Refund request of $249.99 exceeds autonomous limit of $50.00 (POL-REF-001)
Risk Score: 0.6 | Supervisor Decision: APPROVED


## 4. Benchmark Evaluation Metrics (All 10 Scenarios)
Execute the full evaluation suite and display performance metrics.

In [5]:
eval_results = run_evaluation()
pd.DataFrame([eval_results['metrics']]).T.rename(columns={0: 'Metric Value'})

STARTING CUSTOMER SUPPORT TICKET TRIAGE BENCHMARK EVALUATION (10 SCENARIOS)

[1/10] Running SCENARIO-01: Standard Autonomous Refund...
    Category: Billing & Refunds (✓) | Approval: False (✓) | Action: issue_refund (✓) | Latency: 7.6ms

[2/10] Running SCENARIO-02: High-Value Hardware Return Exceeding Threshold...
    Category: Billing & Refunds (✓) | Approval: True (✓) | Action: issue_refund (✓) | Latency: 7.8ms

[3/10] Running SCENARIO-03: Critical Account Compromise & 2FA Reset...
    Category: Account & Security (✓) | Approval: True (✓) | Action: escalate_security_and_freeze_sessions (✓) | Latency: 8.9ms

[4/10] Running SCENARIO-04: P2 Technical Bug - Mobile Checkout Crash Loop...
    Category: Technical Support (✓) | Approval: False (✓) | Action: file_engineering_bug_and_workaround (✓) | Latency: 7.3ms

[5/10] Running SCENARIO-05: VIP Platinum Delayed Courier Shipment...
    Category: Shipping & Logistics (✓) | Approval: True (✓) | Action: expedited_courier_trace_and_credit (✓) | 


Artifacts successfully written to:
  - C:\Users\ROSHAN ALEXANDER\Documents\Projects\Customer-Support-Ticket-Triage\results\evaluation_results.json
  - C:\Users\ROSHAN ALEXANDER\Documents\Projects\Customer-Support-Ticket-Triage\results\10_scenarios_traces.json
  - C:\Users\ROSHAN ALEXANDER\Documents\Projects\Customer-Support-Ticket-Triage\results\evaluation_summary.md
  - C:\Users\ROSHAN ALEXANDER\Documents\Projects\Customer-Support-Ticket-Triage\results\figures\evaluation_dashboard.png


,Metric Value
total_scenarios,10.00
scenarios_passed,10.00
scenario_pass_rate_pct,100.00
category_accuracy_pct,100.00
urgency_accuracy_pct,100.00
sentiment_accuracy_pct,100.00
human_approval_gate_accuracy_pct,100.00
policy_retrieval_hit_rate_pct,100.00
action_execution_alignment_pct,100.00
execution_success_rate_pct,100.00


## 5. Audit Trail Inspection
Inspect the immutable step-by-step telemetry trace from Scenario 1.

In [6]:
trace_df = pd.DataFrame(res1['audit_trail'])[['step', 'agent', 'action', 'duration_ms', 'status']]
trace_df

,step,agent,action,duration_ms,status
0,CLASSIFY_TICKET,TicketClassificationAgent,classify_ticket_content,0.34,SUCCESS
1,CRM_LOOKUP_TOOL,KnowledgeRetrievalAgent,query_customer_profile,0.00,SUCCESS
2,POLICY_RETRIEVER_TOOL,KnowledgeRetrievalAgent,search_policy_knowledge_base,1.07,SUCCESS
3,DRAFT_RESOLUTION,ResolutionAgent,propose_resolution_and_draft_response,0.01,SUCCESS
4,EVALUATE_ESCALATION,EscalationAgent,evaluate_safety_boundaries,0.01,SUCCESS
5,EXECUTE_ACTION_TOOL,ActionExecutionEngine,execute_issue_refund,0.03,SUCCESS
6,FINALIZE_TICKET,WorkflowOrchestrator,finalize_workflow_and_persist_state,0.90,SUCCESS
